# 3D Volumetric Segmentation — Exploration

This notebook explores 3D volumetric data and demonstrates the segmentation pipeline:
- Visualise sample volumes and masks
- Inspect model architecture
- Analyse Dice and IoU metrics after training

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import torch
from model import UNet3D
from metrics import DiceScore, IoUScore

In [ ]:
# Inspect model architecture and parameter count
model = UNet3D(in_channels=1, out_channels=1)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'3D U-Net trainable parameters: {n_params:,}')
print(model)

In [ ]:
# Generate a synthetic volume and visualise 3 orthogonal slices
volume = np.random.rand(64, 64, 64).astype(np.float32)
mask   = (volume > 0.7).astype(np.float32)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
mid = volume.shape[0] // 2

axes[0].imshow(volume[mid, :, :], cmap='gray')
axes[0].set_title('Axial slice (XY)')

axes[1].imshow(volume[:, mid, :], cmap='gray')
axes[1].set_title('Coronal slice (XZ)')

axes[2].imshow(volume[:, :, mid], cmap='gray')
axes[2].set_title('Sagittal slice (YZ)')

for ax in axes:
    ax.axis('off')
plt.suptitle('Synthetic 3D Volume — Orthogonal Views', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate Dice and IoU computation
pred  = torch.tensor(mask).unsqueeze(0).unsqueeze(0)  # (1,1,D,H,W)
target = torch.tensor(mask).unsqueeze(0).unsqueeze(0)

dice_fn = DiceScore()
iou_fn  = IoUScore()

print(f'Dice (perfect prediction): {dice_fn(pred, target):.4f}')
print(f'IoU  (perfect prediction): {iou_fn(pred, target):.4f}')

# With noise
noisy_pred = torch.clamp(pred + torch.randn_like(pred) * 0.3, 0, 1)
print(f'Dice (noisy prediction):   {dice_fn(noisy_pred, target):.4f}')
print(f'IoU  (noisy prediction):   {iou_fn(noisy_pred, target):.4f}')